# Trading App v2: Graph Oracle $1T Smoke Test

This experiment builds all entry/exit edges for a one-year, $1T universe. It derives separate long-entry and short-entry graph-oracle labels, compares them with the existing DP oracle, and runs a small classifier smoke test. The labels are intentionally oracle labels; production lookahead controls are outside this experiment.

In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'pyproject.toml').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
WORKSPACE_ROOT = REPO_ROOT.parent
for package, repo in [('quant_warehouse', WORKSPACE_ROOT / 'quant-warehouse'), ('quant_orchestrator', WORKSPACE_ROOT / 'quant-orchestrator')]:
    if repo.exists() and str(repo) not in sys.path:
        sys.path.insert(0, str(repo))

from quant_warehouse.research_tools import FamilyEvaluationConfig, screen_fmp_equity_universe
from quant_warehouse.platforms.data_providers.fmp.target_engineering import LabelBuildSpec, build_trade_results
from quant_warehouse.warehouse.api import Warehouse

MIN_MARKET_CAP = 1_000_000_000_000
START_DATE = os.getenv('GRAPH_ORACLE_START_DATE', '2025-01-01')
END_DATE = os.getenv('GRAPH_ORACLE_END_DATE', '2025-12-31')
MAX_SYMBOLS = int(os.getenv('GRAPH_ORACLE_MAX_SYMBOLS', '0'))  # 0 = all screened symbols
MIN_RETURN = 0.01
MAX_HOLDING_DAYS = 120
SCORE_QUANTILE = 0.80
RANDOM_SEED = 20260715


## Load the $1T price universe

The smoke test uses cached FMP prices. Set `GRAPH_ORACLE_MAX_SYMBOLS` to a small number when iterating locally.

In [ ]:
warehouse = Warehouse()
feature_config = FamilyEvaluationConfig(market_cap_min=MIN_MARKET_CAP, start_date=START_DATE, end_date=END_DATE)
symbols, raw_universe, eligibility, universe_source = screen_fmp_equity_universe(
    feature_config, warehouse=warehouse, required_sections=('prices',)
)
symbols = [str(symbol).upper() for symbol in symbols]
if MAX_SYMBOLS:
    symbols = symbols[:MAX_SYMBOLS]
price_frames = {}
for symbol in symbols:
    frame = warehouse.read_prices(symbol, provider='fmp', start=START_DATE, end=END_DATE)
    if frame is not None and not frame.empty:
        frame = frame.copy().reset_index(names='date')
        frame['date'] = pd.to_datetime(frame['date'], errors='coerce').dt.normalize()
        frame = frame.dropna(subset=['date']).sort_values('date').drop_duplicates('date')
        if {'high', 'low'}.issubset(frame.columns) and len(frame) >= 30:
            price_frames[symbol] = frame
symbols = sorted(price_frames)
print({'universe_source': universe_source, 'symbols': len(symbols), 'start': START_DATE, 'end': END_DATE})
display(pd.DataFrame([{'symbol': s, 'rows': len(price_frames[s]), 'first': price_frames[s]['date'].min(), 'last': price_frames[s]['date'].max()} for s in symbols]))

## Build the all-pairs graph oracle

For every `i < j`, the long graph contains the return from buying at `i` and selling at `j`; the short graph contains the return from shorting at `i` and covering at `j`. Entry scores aggregate profitable outgoing edges. Exit scores aggregate profitable incoming edges.

In [ ]:
def graph_oracle_for_symbol(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    work = frame.sort_values('date').reset_index(drop=True).copy()
    dates = work['date'].to_numpy()
    buy = pd.to_numeric(work['high'], errors='coerce').to_numpy(float)
    sell = pd.to_numeric(work['low'], errors='coerce').to_numpy(float)
    n = len(work)
    long_return = sell[None, :] / buy[:, None] - 1.0
    short_return = buy[:, None] / sell[None, :] - 1.0
    forward = np.triu(np.ones((n, n), dtype=bool), k=1)
    horizon = np.arange(n)[None, :] - np.arange(n)[:, None]
    valid = forward & (horizon <= MAX_HOLDING_DAYS)
    def side_metrics(returns: np.ndarray, prefix: str) -> tuple[pd.DataFrame, pd.DataFrame]:
        weights = np.where(valid, np.maximum(returns - MIN_RETURN, 0.0), 0.0)
        positive = (weights > 0).astype(float)
        hub = np.ones(n, dtype=float)
        authority = np.ones(n, dtype=float)
        for _ in range(30):
            authority = weights.T @ hub
            authority = authority / (np.linalg.norm(authority) or 1.0)
            hub = weights @ authority
            hub = hub / (np.linalg.norm(hub) or 1.0)
        entry = pd.DataFrame({
            'date': dates,
            f'{prefix}_entry_score': weights.sum(axis=1),
            f'{prefix}_positive_exit_count': positive.sum(axis=1),
            f'{prefix}_best_return': np.where(valid, returns, np.nan).max(axis=1),
            f'{prefix}_entry_hits_score': hub,
        })
        exit_ = pd.DataFrame({
            'date': dates,
            f'{prefix}_exit_score': weights.sum(axis=0),
            f'{prefix}_positive_entry_count': positive.sum(axis=0),
            f'{prefix}_exit_hits_score': authority,
        })
        return entry, exit_
    long_entry, long_exit = side_metrics(long_return, 'long')
    short_entry, short_exit = side_metrics(short_return, 'short')
    labels = long_entry.merge(short_entry, on='date').merge(long_exit, on='date').merge(short_exit, on='date')
    for side in ('long', 'short'):
        score = labels[f'{side}_entry_score']
        threshold = score[score > 0].quantile(SCORE_QUANTILE) if (score > 0).any() else np.inf
        labels[f'graph_{side}_label'] = (score >= threshold).astype('uint8')
    edges = []
    for i, j in zip(*np.where(valid & ((long_return >= MIN_RETURN) | (short_return >= MIN_RETURN)))):
        edges.append({'entry_date': dates[i], 'exit_date': dates[j], 'holding_days': int(horizon[i, j]),
                      'long_return': long_return[i, j], 'short_return': short_return[i, j]})
    return labels, pd.DataFrame(edges)

label_parts, edge_parts = [], []
for symbol, frame in price_frames.items():
    labels, edges = graph_oracle_for_symbol(frame)
    labels.insert(0, 'symbol', symbol)
    edges.insert(0, 'symbol', symbol)
    label_parts.append(labels)
    edge_parts.append(edges)
graph_labels = pd.concat(label_parts, ignore_index=True) if label_parts else pd.DataFrame()
graph_edges = pd.concat(edge_parts, ignore_index=True) if edge_parts else pd.DataFrame()
print({'graph_label_rows': len(graph_labels), 'positive_edges': len(graph_edges), 'edge_columns': list(graph_edges.columns)})
display(graph_labels.filter(regex='symbol|date|entry_score|label').head(20))
display(graph_edges.groupby('symbol').size().describe() if not graph_edges.empty else pd.DataFrame())

## Compare against the current DP oracle

DP produces a constrained non-overlapping schedule. Here we mark its selected entry dates as long or short labels and compare those sparse labels with the graph labels.

In [ ]:
dp_spec = LabelBuildSpec(
    k_params={'YE': [1, 3]}, min_profit_pct=MIN_RETURN,
    buy_execution='high', sell_execution='low', short_execution='low', cover_execution='high',
    start_date=START_DATE, end_date=END_DATE,
)
dp_result = build_trade_results(symbols, spec=dp_spec, price_frames=price_frames)
dp_trades = pd.DataFrame(dp_result.completed_trades)
dp_labels = graph_labels[['symbol', 'date']].copy()
dp_labels['dp_long_label'] = 0
dp_labels['dp_short_label'] = 0
if not dp_trades.empty:
    dp_trades['symbol'] = dp_trades['symbol'].astype(str).str.upper()
    dp_trades['entry_date'] = pd.to_datetime(dp_trades['entry_date'], errors='coerce').dt.normalize()
    for side, column in [('long', 'dp_long_label'), ('short', 'dp_short_label')]:
        keys = dp_trades.loc[dp_trades['side'].astype(str).str.lower().eq(side), ['symbol', 'entry_date']].drop_duplicates()
        key_set = set(zip(keys['symbol'], keys['entry_date']))
        dp_labels[column] = [int((symbol, date) in key_set) for symbol, date in zip(dp_labels['symbol'], dp_labels['date'])]
comparison = graph_labels.merge(dp_labels, on=['symbol', 'date'], how='left')
rows = []
for side in ('long', 'short'):
    g, d = f'graph_{side}_label', f'dp_{side}_label'
    rows.append({'side': side, 'graph_positives': int(comparison[g].sum()), 'dp_positives': int(comparison[d].sum()),
                  'intersection': int((comparison[g].astype(bool) & comparison[d].astype(bool)).sum()),
                  'graph_rate': float(comparison[g].mean()), 'dp_rate': float(comparison[d].mean())})
display(pd.DataFrame(rows))
display(comparison.groupby('symbol')[['graph_long_label', 'graph_short_label', 'dp_long_label', 'dp_short_label']].sum().head(30))

## Classifier smoke test

This is deliberately lightweight: price-derived features are used only to verify that the graph targets can be consumed by a long/short classifier. The production feature-family panel can replace these features in the next experiment.

In [ ]:
feature_parts = []
for symbol, frame in price_frames.items():
    x = frame[['date', 'close']].copy() if 'close' in frame else frame[['date', 'high']].rename(columns={'high': 'close'})
    x['symbol'] = symbol
    px = pd.to_numeric(x['close'], errors='coerce')
    x['ret_5'] = px.pct_change(5)
    x['ret_20'] = px.pct_change(20)
    x['ret_60'] = px.pct_change(60)
    x['vol_20'] = px.pct_change().rolling(20).std()
    x['dist_ma_20'] = px / px.rolling(20).mean() - 1.0
    feature_parts.append(x.drop(columns='close'))
features = pd.concat(feature_parts, ignore_index=True).dropna()
model_data = features.merge(comparison, on=['symbol', 'date'], how='inner')
cutoff = model_data['date'].quantile(0.70)
train = model_data[model_data['date'] <= cutoff]
test = model_data[model_data['date'] > cutoff]
feature_cols = ['ret_5', 'ret_20', 'ret_60', 'vol_20', 'dist_ma_20']
metrics = []
for side in ('long', 'short'):
    target = f'graph_{side}_label'
    if train[target].nunique() < 2 or test[target].nunique() < 2:
        metrics.append({'side': side, 'status': 'insufficient_class_support'})
        continue
    model = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, class_weight='balanced_subsample', random_state=RANDOM_SEED, n_jobs=-1)
    model.fit(train[feature_cols], train[target])
    probability = model.predict_proba(test[feature_cols])[:, 1]
    metrics.append({'side': side, 'status': 'ok', 'train_rows': len(train), 'test_rows': len(test),
                    'train_positive_rate': float(train[target].mean()), 'test_positive_rate': float(test[target].mean()),
                    'test_roc_auc': float(roc_auc_score(test[target], probability)),
                    'test_average_precision': float(average_precision_score(test[target], probability))})
display(pd.DataFrame(metrics))
display(pd.DataFrame({'feature': feature_cols, 'importance': model.feature_importances_}).sort_values('importance', ascending=False) if 'model' in locals() else pd.DataFrame())

## Interpretation

The graph label is a measure of future optionality: how many profitable exits and how much aggregate profitable return were available from an entry date. A strong overlap with DP means the graph is rediscovering DP's selected opportunities; a low overlap with useful classifier performance means it may provide a broader learning target. This smoke test is not a trading result until predicted scores are passed through the same non-overlapping execution/backtest machinery as the DP baseline.